# Marketing Spend and Sales: Linear Regression Analysis

## Business Objective

This project examines the relationship between marketing promotional spending and sales revenue across TV, radio, and social media campaigns.

The objective is to identify which marketing channel has the strongest linear relationship with sales and quantify that relationship using ordinary least squares (OLS) regression.

The analysis covers data cleaning, exploratory data analysis (EDA), model selection, regression diagnostics, coefficient interpretation, and business recommendations.


## 1. Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.formula.api import ols

sns.set_theme(style="whitegrid")


## 2. Load the Dataset

The dataset contains promotional spending for TV, radio, and social media, along with sales generated by each marketing promotion.

Place `marketing_and_sales_data_evaluate_lr.csv` in the same folder as this notebook before running the following cell.


In [ ]:
data = pd.read_csv("marketing_and_sales_data_evaluate_lr.csv")

print("Dataset shape:", data.shape)
display(data.head())


## 3. Data Overview

Before modeling, I inspect the structure of the dataset, descriptive statistics, and missing values.


In [ ]:
display(data.info())


In [ ]:
display(data[["TV", "Radio", "Social_Media", "Sales"]].describe())


In [ ]:
missing = data.isna().sum().sort_values(ascending=False)
missing_pct = (data.isna().mean() * 100).sort_values(ascending=False)

missing_summary = pd.DataFrame({
    "Missing Values": missing,
    "Missing Percentage": missing_pct.round(2)
})

display(missing_summary)


### Handling Missing Sales Values

Rows without a sales value cannot contribute to a regression model in which `Sales` is the dependent variable, so those observations are removed.


In [ ]:
data = data.dropna(subset=["Sales"]).copy()

print("Shape after removing missing Sales values:", data.shape)


## 4. Exploratory Data Analysis

EDA helps assess distributions and visualize the relationships between the marketing variables and sales before fitting a model.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(data["Sales"], kde=True, ax=ax)
ax.set_title("Distribution of Sales")
ax.set_xlabel("Sales")
ax.set_ylabel("Frequency")
plt.show()


In [ ]:
sns.pairplot(data[["TV", "Radio", "Social_Media", "Sales"]])
plt.show()


### Selecting the Predictor

The pairwise relationships suggest that **TV promotional spending has the strongest positive linear relationship with Sales**. Radio also shows a positive relationship, but with greater dispersion, while the relationship between Social Media spending and Sales appears comparatively weak.

Based on this visual assessment, `TV` is selected as the independent variable for the simple linear regression model.


## 5. Simple Linear Regression

The model is specified as:

**Sales = β₀ + β₁(TV) + ε**

where `Sales` is the dependent variable and `TV` is the independent variable.


In [ ]:
ols_formula = "Sales ~ TV"

ols_model = ols(formula=ols_formula, data=data)
model = ols_model.fit()

print(model.summary())


## 6. Model Diagnostics

A linear regression model relies on several assumptions. I assess:

1. **Linearity** — the relationship between TV spending and Sales should be approximately linear.
2. **Independence** — observations should be independent.
3. **Normality of residuals** — residuals should be approximately normally distributed.
4. **Homoscedasticity** — residual variance should remain reasonably constant across fitted values.


### 6.1 Linearity

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(x=data["TV"], y=data["Sales"])
plt.title("TV Promotional Spending vs. Sales")
plt.xlabel("TV Promotional Budget")
plt.ylabel("Sales")
plt.show()


The scatterplot shows a pronounced linear relationship between TV promotional spending and Sales, supporting the linearity assumption.


### 6.2 Independence

Each row represents a separate marketing promotion. Based on the structure of the provided dataset, the observations are treated as independent for this analysis.


### 6.3 Normality of Residuals

In [ ]:
residuals = model.resid

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(residuals, kde=True, ax=axes[0])
axes[0].set_title("Histogram of Residuals")
axes[0].set_xlabel("Residual")

sm.qqplot(residuals, line="s", ax=axes[1])
axes[1].set_title("Normal Q-Q Plot")

plt.tight_layout()
plt.show()


The residual histogram is approximately bell-shaped, and the Q-Q plot is close to a straight line. Together, these plots provide visual support for approximate normality of the residuals.


### 6.4 Homoscedasticity

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(x=model.fittedvalues, y=model.resid)
plt.axhline(0, linestyle="--")
plt.title("Fitted Values vs. Residuals")
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.show()


The residuals appear to maintain a relatively consistent spread across fitted values, providing visual support for the homoscedasticity assumption.


## 7. Model Results and Interpretation

In [ ]:
results = pd.DataFrame({
    "Coefficient": model.params,
    "P-value": model.pvalues,
    "CI Lower": model.conf_int()[0],
    "CI Upper": model.conf_int()[1]
})

display(results)
print(f"R-squared: {model.rsquared:.4f}")
print(f"Adjusted R-squared: {model.rsquared_adj:.4f}")


### Key Findings

For the provided dataset, the simple linear regression using TV promotional spending as the predictor produces an **R-squared of approximately 0.999**.

The estimated regression equation is:

**Sales = -0.1263 + 3.5614 × TV**

The TV coefficient is approximately **3.5614**, with a p-value below conventional significance thresholds. The reported 95% confidence interval is approximately **[3.558, 3.565]**.

Within this model and dataset, a one-unit increase in TV promotional spending is associated with an estimated 3.5614-unit increase in Sales.


## 8. Business Interpretation

TV promotional spending exhibits an exceptionally strong linear association with sales in this dataset. The model therefore identifies TV as a particularly strong predictor of Sales among the three available promotional channels.

From a business perspective, the analysis suggests that TV spending deserves close consideration when allocating future promotional budgets.

However, the regression identifies **association rather than causation**. A high R-squared does not, by itself, establish that increasing TV expenditure will necessarily cause the same proportional increase in sales in a real-world setting.


## 9. Recommendations

- Prioritize TV as a channel for further investigation because it has the strongest observed linear relationship with Sales.
- Use the regression equation as an analytical benchmark for estimating sales associated with different TV budgets.
- Evaluate TV alongside Radio and Social Media in a multiple-regression framework before making major budget-allocation decisions.
- Consider additional factors, such as campaign quality, seasonality, audience characteristics, and market conditions, before treating the estimated relationship as a causal business rule.


## 10. Limitations and Further Analysis

This analysis is intentionally focused on simple linear regression. A stronger extension would compare the TV-only model with a multiple linear regression model incorporating `TV`, `Radio`, and `Social_Media` simultaneously.

Further work could also include outlier analysis, formal heteroscedasticity tests, train-test validation, prediction intervals, and evaluation of whether the linear specification remains appropriate outside the observed range of the data.


## 11. Conclusion

The exploratory analysis indicates that TV promotional spending has the strongest positive linear relationship with Sales in the provided dataset. The simple OLS model explains approximately 99.9% of the observed variation in Sales, and the diagnostic plots provide visual support for the key regression assumptions.

The results make TV a strong candidate for further analysis and business consideration, while the limitations of observational regression should be kept in mind when translating the findings into budget decisions.
